# Tutorial 3.3: Attribution analysis

In [ ]:
import os, warnings, torch

import numpy as np
import scanpy as sc
import pandas as pd

from datasets.data_manager_SMA import SMA
from utils.notebook_graph_utils import build_graph_model, feature_gradients, load_graph_checkpoint
from utils.utils import *
from utils.utils_dataloader import *
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from collections import defaultdict

from palettable.cartocolors.diverging import *
from palettable.scientific.diverging import *


### Load dataset

In [ ]:
# please load the target Visium data (from the uploaded)
adata_path = '/mnt/datadisk0/Processed_DATA/2023_nbt_SMA/Processed_data_used/V11L12-109_A1'
rna_adata = sc.read_visium(path=adata_path, count_file='filtered_feature_bc_matrix.h5')

In [ ]:
coordiantes = []
for i in range(rna_adata.shape[0]):
    coordiantes.append(str(rna_adata.obs['array_row'].values[i]) + '_' + str(rna_adata.obs['array_col'].values[i]) )
coordiantes = np.array(coordiantes)

rna_adata.obs['coordinates'] = coordiantes

### Load args

In [ ]:
%run ./args/args_SMA.py
args = args

### Create dataloader

In [ ]:
# create the dataset
dataset = SMA(
    path_img=args.path_img,
    rna_path=args.rna_path,
    msi_path=args.msi_path,
    n_top_genes=args.n_source,
    n_top_targets=args.n_target,
    graph_k=args.graph_k,
    val_ratio=args.val_ratio,
    mask_seed=args.seed,
)


### Model initialization

In [ ]:
# create the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_graph_model(args, dataset, use_cell_type=False).to(device)
model = load_graph_checkpoint(model, 'WholeSliceGraphTransformer_SMA_last.pth', device=device)


### Model inference 

In [ ]:
gradients, outputs, mask = feature_gradients(
    model,
    dataset.testing[0],
    target_index=0,
    split='test',
    device=device,
)
attributions = gradients.numpy()


In [ ]:
num_genes = 30
attributions = np.abs(attributions)
grad_norm = attributions.sum(axis=0)


In [ ]:
top_indices = np.argsort(grad_norm)[-num_genes:]
temp_mask = grad_norm >= grad_norm[top_indices[0]]

highly_correlated_genes_0 = dataset.source_panel[top_indices]


In [ ]:
sorted_data = np.sort(grad_norm.squeeze())[::-1]
indices = np.arange(len(sorted_data))

fig, ax = plt.subplots(1, figsize=(6.5, 2), dpi=300)

plt.scatter(indices[0:30], sorted_data[0:30], s=2, color='#6F1D57')
plt.scatter(29 + (indices[30:] -29)/40, sorted_data[30:], s=0.2, color='#FCBD8B')

num_genes
for i in range(num_genes):
    plt.annotate(highly_correlated_genes_0[num_genes-i -1],
                 xy=(i, sorted_data[i]),
                 xytext=(0, 5),  
                 textcoords='offset points',
                 ha='center',  
                 rotation=90,
                 fontsize=4.5)  

                 
plt.fill_between(indices[0:30], sorted_data[0:30], color='#E8789A', alpha=0.4)
plt.fill_between(29 + (indices[30:]- 29 )/40, sorted_data[30:], color='#EDDAB9', alpha=0.4)

custom_xticks = [0, 30, 60]
custom_xticklabels = ['0', '30', '1230']

ax.set_xticks(custom_xticks)
ax.set_xticklabels(custom_xticklabels, fontsize=5)
plt.yticks(fontsize=5)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.show()